# El-Bagawat task2 — Phase 0: Data Audit and Environment Setup
This notebook inspects all base data sources to verify their completeness, schemas, layer structures, and coordinate references.

---

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import ezdxf
from shapely.geometry import Point, LineString
import shapely.ops
from skimage.transform import ProjectiveTransform, AffineTransform

# Define base directory paths
    "BASE = Path.cwd().parent\n"    "DATA = BASE / 'data/BaseSiteCAD'\n"
print('Libraries imported successfully.')
print(f'Workspace Root: {BASE}')

## Step 0a — Inspect the DWG / Converted DXF
We inspect the layer structures of the main converted site CAD file to identify entity types, coordinates, and any road/path features.

In [ ]:
    "dxf_path = DATA / 'BaseSiteCAD/Site_CAD_Working_converted.dxf'\n"print(f'Reading DXF from: {dxf_path}')
doc = ezdxf.readfile(dxf_path)
msp = doc.modelspace()

# Count entity types per layer
from collections import Counter
layer_entities = {}
for entity in msp:
    layer = entity.dxf.layer
    etype = entity.dxftype()
    if layer not in layer_entities:
        layer_entities[layer] = Counter()
    layer_entities[layer][etype] += 1

print('\nLayers and Entity Counts:')
for layer, counts in sorted(layer_entities.items()):
    print(f'Layer: {layer}')
    for etype, count in counts.items():
        print(f'  {etype}: {count}')

## Step 0b — Read and Compare the QGIS GCP Files
We compare the three ground control point (GCP) files to identify coordinate agreement and select the set with the highest density of points for georeferencing.

In [ ]:
gcp_dir = DATA / '130_BuildingFootprintsVectorData/BuildingTracesCurrent'
gcp_files = [
    'Buildings_Mask.shp.points',
    'Buildings_Mask.shp.points1.points',
    'Buildings_Mask.shp.points2.points'
]

gcp_sets = {}
for f in gcp_files:
    full_path = gcp_dir / f
    # Use latin-1 encoding to support degree symbol in comments
    df = pd.read_csv(full_path, skiprows=1, encoding='latin-1')
    # Clean column headers
    df.columns = [c.strip().replace('"', '').replace("'", '') for c in df.columns]
    gcp_sets[f] = df
    print(f'Loaded {len(df)} points from {f}')

# Check subset and coordinate agreement
p_ref = gcp_sets['Buildings_Mask.shp.points']
p1 = gcp_sets['Buildings_Mask.shp.points1.points']
p2 = gcp_sets['Buildings_Mask.shp.points2.points']

merged_1 = pd.merge(p_ref, p1, on=['mapX', 'mapY'], suffixes=('_ref', '_1'))
merged_2 = pd.merge(p1, p2, on=['mapX', 'mapY'], suffixes=('_1', '_2'))

max_diff_1 = np.max(np.sqrt((merged_1['sourceX_ref'] - merged_1['sourceX_1'])**2 + (merged_1['sourceY_ref'] - merged_1['sourceY_1'])**2))
max_diff_2 = np.max(np.sqrt((merged_2['sourceX_1'] - merged_2['sourceX_2'])**2 + (merged_2['sourceY_1'] - merged_2['sourceY_2'])**2))

print(f'\nCoordinate Agreement Analysis:')
print(f'  Is ref points a subset of points1? {set(zip(p_ref.mapX, p_ref.mapY)).issubset(set(zip(p1.mapX, p1.mapY)))}')
print(f'  Is points1 a subset of points2? {set(zip(p1.mapX, p1.mapY)).issubset(set(zip(p2.mapX, p2.mapY)))}')
print(f'  Max source pixel shift (ref vs 1): {max_diff_1:.6f} units')
print(f'  Max source pixel shift (1 vs 2): {max_diff_2:.6f} units')
print('\nDecision: Selected Buildings_Mask.shp.points2.points as the authoritative GCP file (32 points).')

## Step 0c — CRS Audit
We inspect coordinate systems of footprints, boundaries, DEMs, and satellite imagery to identify necessary reprojection tasks.

In [ ]:
layers = {
    'footprints': DATA / '130_BuildingFootprintsVectorData/BuildingTracesCurrent/Buildings_Mask.shp',
    'roi_full':   DATA / '110_GISRegionOfInterest/Bagawat_ROI.shp',
    'roi_small':  DATA / '110_GISRegionOfInterest/BagawatROI_Smaller.shp',
    'dem':        DATA / '150_DigitalElevationModel/Generated_DEMs/Current_DEM/Bagawat-DEM-NewImageryOnly-0.4m-DEM.tif',
    'wv2_p001':   DATA / '140_SAR_Imagery/DigitalGlobe_2018/MONO/058239078010_01/058239078010_01_P001_MUL/18JUN14090738-M2AS_R1C1-058239078010_01_P001.TIF'
}

print('CRS Audit Results:')
for name, path in layers.items():
    if path.suffix.lower() == '.shp':
        gdf = gpd.read_file(path)
        print(f'  {name} ({path.name}): CRS = {gdf.crs}')
    else:
        with rasterio.open(path) as src:
            print(f'  {name} ({path.name}): CRS = {src.crs}')

print('\nDecision: Establish EPSG:32636 as Working CRS. Reproject geographic ROI vectors (EPSG:4326) to EPSG:32636.')

## Step 0d — Parse Individual DXF Files and Verify Alignments
We parse the 7 individual calibration building files and verify they coincide with the shapefile footprints when local building-specific translations are applied.

In [ ]:
dxf_files = {
    1: 'Building1.dxf',
    23: 'Building23.dxf',
    24: 'Building24.dxf',
    25: 'Building25.dxf',
    26: 'Building26.dxf',
    175: 'Building175.dxf',
    210: 'Building210.dxf'
}
dxf_base = DATA / '120_SiteReport/BaseSiteCAD'
footprints = gpd.read_file(layers['footprints'])

def get_dxf_geom(entity):
    etype = entity.dxftype()
    if etype == 'LWPOLYLINE':
        pts = list(entity.get_points(format='xy'))
        if len(pts) >= 2: return LineString(pts)
    elif etype == 'POLYLINE':
        pts = [v.dxf.location for v in entity.vertices]
        if len(pts) >= 2: return LineString(pts)
    elif etype == 'LINE':
        return LineString([entity.dxf.start, entity.dxf.end])
    return None

print('Individual DXF Spatial Overlap Verification:')
for bld_id, filename in dxf_files.items():
    path = dxf_base / filename
    doc = ezdxf.readfile(path)
    msp = doc.modelspace()
    
    # Get shapefile geometry
    fp_geom = footprints[footprints['ID'] == bld_id].iloc[0].geometry
    centroid = fp_geom.centroid
    
    # Find matching label in DXF
    label_x, label_y = None, None
    for entity in msp:
        if entity.dxf.layer == 'NUMBERING' and entity.dxftype() == 'TEXT':
            text_val = entity.dxf.text.strip()
            if text_val == str(bld_id) or (bld_id == 25 and 'peace' in text_val.lower()):
                label_x = entity.dxf.insert.x
                label_y = entity.dxf.insert.y
                break
                
    # Fallback to median coordinates if text not found
    if label_x is None:
        xs, ys = [], []
        for entity in msp:
            if entity.dxftype() in ('LINE', 'LWPOLYLINE', 'POLYLINE'):
                g = get_dxf_geom(entity)
                if g:
                    xs.extend([p[0] for p in g.coords])
                    ys.extend([p[1] for p in g.coords])
        if xs:
            label_x = np.median(xs)
            label_y = np.median(ys)
            
    # Compute local translation offset
    tx = centroid.x - 0.001 * label_x
    ty = centroid.y - 0.001 * label_y
    
    # Apply translation and calculate overlap with footprint
    translated_lines = []
    for entity in msp:
        if entity.dxf.layer in ('LW1', 'LW2') and entity.dxftype() in ('LWPOLYLINE', 'POLYLINE', 'LINE'):
            g = get_dxf_geom(entity)
            if g:
                coords = [(0.001 * x + tx, 0.001 * y + ty) for x, y in g.coords]
                translated_lines.append(LineString(coords))
                
    translated_union = shapely.ops.unary_union(translated_lines)
    min_dist = fp_geom.distance(translated_union)
    buffered = translated_union.buffer(1.5)
    overlap_pct = (fp_geom.intersection(buffered).area / fp_geom.area) * 100
    
    print(f'  Building {bld_id}: Dist={min_dist:.4f} m, Overlap={overlap_pct:.1f}%')

## Step 0e — Inspect Excel Database
We inspect the corrected Excel database `2026 El Bagawat Database Draft 1.xlsx` to check the sheet structures and data availability for building entrance directions.

In [ ]:
    "excel_path = DATA / 'BaseSiteCAD/2026 El Bagawat Database Draft 1.xlsx'\n"print(f'Reading Excel file: {excel_path}')

# Load sheets
xl = pd.ExcelFile(excel_path)
print(f'Sheets: {xl.sheet_names}')

# Parse main database
db = xl.parse('Database Full')
print(f'Database Full Row/Col Dimensions: {db.shape}')

# Display value counts for entrance directions
dir_col = 'Entrace Direction'
if dir_col in db.columns:
    print(f'\nEntrance Direction values distribution in \'{dir_col}\':')
    print(db[dir_col].value_counts(dropna=False))
else:
    print(f'Column \'{dir_col}\' not found in columns: {list(db.columns)[:5]}')